# General Database Connection

The first two cells import python libraries for handling data and database  
connections. The next cell establishes a connection to the ODOT data  
warehouse using your Windows credentials. It stores this connection in the 
conn object.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
connection_string = (
    r"Driver=SQL Server;"
    r"Server=DOTWarehousePrd;"
    r"Database=Warehouse;"
    r"Trusted_Connection=yes;"
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)
conn = engine.connect()

# Recreating the "Initial Bridge List" Tab

In [3]:
# ---------------------------------------------------------
# 2. RAW SQL QUERY (REPLICATING ACCESS QUERY 1)
# ---------------------------------------------------------
# Note: Column names and order match the Access output for easy auditing.
sql_initial_bridge_list = text("""
SELECT DISTINCT 
    p.PROPOSAL_ID, 
    c.PROJECT_ID, 
    ps.PROPOSALSECTION_ID, 
    l.LETTING_DT, 
    l.LETTING_NM, 
    p.PROPOSAL_NM, 
    -- Mid([PROPOSAL_NM],4,Len([PROPOSAL_NM])-3) AS PID
    SUBSTRING(p.PROPOSAL_NM, 4, LEN(p.PROPOSAL_NM) - 3) AS PID, 
    p.CONTRACTALTERNATE_NM1, 
    p.AWARDEDAMOUNT, 
    p.FEDPROJECTNUM, 
    ps.PROPOSALSECTION_NM, 
    ps.DESCR AS [Section Desc], 
    c.ESSST3 AS [Existing SFN], 
    c.ESSST4 AS [New SFN], 
    c.FEDWORKCLASS, 
    bs.BRIDGE_NM, 
    bs.DESCR, 
    bs.TYPE, 
    bs.LENGTH, 
    bs.WIDTH, 
    bs.NUMOFSPANS, 
    'NO' AS Modified, 
    'NO' AS [Flag for Removal]
FROM dbo.PRECON_LETTING l
INNER JOIN dbo.PRECON_PROPOSAL p ON l.LETTING_ID = p.LETTING_ID
INNER JOIN dbo.PRECON_PROPOSALSECTION ps ON p.PROPOSAL_ID = ps.PROPOSAL_ID
INNER JOIN dbo.PRECON_PROJECTITEM pi ON ps.PROPOSALSECTION_ID = pi.PROPOSALSECTION_ID
INNER JOIN dbo.PRECON_CATEGORY c ON pi.CATEGORY_ID = c.CATEGORY_ID
-- Left join is used to ensure we keep bridges even if dimensions (bs) are missing
LEFT JOIN dbo.PRECON_BRIDGESEGMENT bs ON c.CATEGORY_ID = bs.CATEGORY_ID
WHERE 
    -- Letting Dates: FFY 2025 Cycle
    l.LETTING_DT >= '2024-10-01' AND l.LETTING_DT <= '2025-09-30'
    AND p.AWARDEDAMOUNT > 0 
    -- Work Class Filter (08-12)
    AND c.FEDWORKCLASS IN ('08', '09', '10', '11', '12')
    -- Category Filter (Structures 14/15)
    AND (LEFT(c.CATEGORY_NM, 2) = '14' OR LEFT(c.CATEGORY_NM, 2) = '15')
ORDER BY 
    l.LETTING_DT, 
    p.PROPOSAL_NM, 
    ps.PROPOSALSECTION_NM;
""")

In [4]:
with engine.connect() as conn:
    initial_bridge_list = pd.read_sql(sql_initial_bridge_list, conn)

initial_bridge_list.sort_values('LETTING_DT')

,PROPOSAL_ID,PROJECT_ID,PROPOSALSECTION_ID,LETTING_DT,LETTING_NM,PROPOSAL_NM,PID,CONTRACTALTERNATE_NM1,AWARDEDAMOUNT,FEDPROJECTNUM,...,New SFN,FEDWORKCLASS,BRIDGE_NM,DESCR,TYPE,LENGTH,WIDTH,NUMOFSPANS,Modified,Flag for Removal
0,9568.0,18200.0,103337.0,2024-10-17,24-10-17,FUL101140,101140,240464,3898020.95,E200(109),...,2601746,10,FUL-120-1408,STATE ROUTE 120 OVER TEN MILE CREEK,X030,36.37,60.3300,1.0,NO,NO
1,9582.0,18178.0,103433.0,2024-10-17,24-10-17,ROS116074,116074,244020,586885.00,E230(721),...,7104937,10,None,None,None,NaN,NaN,NaN,NO,NO
2,9642.0,18318.0,103906.0,2024-11-14,24-11-14,CLA102707,102707,240515,938819.94,E161(411),...,1202139,10,CLA-42-0820,STATE ROUTE 42 OVER GILROY DITCH.,X081,48.63,44.0000,1.0,NO,NO
3,9540.0,18165.0,103123.0,2024-11-14,24-11-14,CUY13184,13184,240510,4861698.83,E191(568),...,1801930,10,1801930,STATE ROUTE 4 OVER TINKERS CREEK,X081,119.67,45.0000,1.0,NO,NO
4,9540.0,18166.0,103123.0,2024-11-14,24-11-14,CUY13184,13184,240510,4861698.83,E191(568),...,1801930,10,1801930,STATE ROUTE 14 OVER TINKERS CREEK,X081,119.67,45.0000,1.0,NO,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,10212.0,19168.0,108532.0,2025-09-11,25-09-11,WOO107711,107711,250399,3693159.78,E210(245),...,None,10,None,None,None,NaN,NaN,NaN,NO,NO
80,10212.0,19169.0,108532.0,2025-09-11,25-09-11,WOO107711,107711,250399,3693159.78,E210(245),...,None,10,None,None,None,NaN,NaN,NaN,NO,NO
82,10252.0,19222.0,108889.0,2025-09-25,25-09-25,BRO114529,114529,250429,764901.05,E240(898),...,0802523,10,None,None,None,NaN,NaN,NaN,NO,NO
81,10263.0,19248.0,108974.0,2025-09-25,25-09-25,BRO100897,100897,250428,5107112.19,E170(246),...,0801120,10,BRO-52-12.452,US-52 OVER RED OAK CREEK.,X040,248.13,48.6667,3.0,NO,NO


# Working Backwards

Next Sheet after Initial Bridge List is "Bridge Types" in python, this is  
just a dictionary;

In [5]:
bridge_types = {
    "X0CB": "CABLE STAY BRIDGE OVER WATER",
    "X002": "TIMBER TRUSS BRIDGE",
    "X020": "CONCRETE SLAB OVER WATERWAY",
    "X028": "BOX CULVERT",
    "X030": "THREE SIDED CULVERT",
    "X031": "STEEL GIRDER BRIDGE OVER WATERWAY",
    "X032": "STEEL TRUSS BRIDGE OVER WATER",
    "X040": "STEEL BEAM BRIDGE OVER WATERWAY",
    "X081": "PRESTRESSED BRIDGE OVER WATERWAY",
    "X120": "CONCRETE SLAB BRIDGE OVER RAILROAD",
    "X131": "STEEL GIRDER BRIDGE OVER HIGHWAY",
    "X135": "STEEL BEAM BRIDGE OVER HIGHWAY",
    "X141": "STEEL BRIDGE OVER RAILROAD",
    "X150": "STEEL BEAM BRIDGE OVER RAILROAD",
    "X181": "PRESTRESSED BRIDGE OVER RAILROAD",
    "X220": "CONCRETE SLAB OVER HIGHWAY",
    "X281": "PRESTRESSED BRIDGE OVER HIGHWAY",
    "X320": "CONCRETE SLAB BRIDGE OVER WATER & RAIL",
    "X331": "STEEL GIRDER BRIDGE OVER WATER & RAIL",
    "X340": "STEEL BEAM BRIDGE OVER WATER AND RAIL",
    "X381": "PRESTRESSED BRIDGE OVER WATER& RAIL",
    "X420": "CONCRETE SLAB BRIDGE OVER WATER & HWY",
    "X481": "PRESTRESSED BRIDGE OVER WATER & HWY",
    "X520": "CONCRETE SLAB BRIDGE OVER RAIL & HWY",
    "X531": "STEEL GIRDER BRIDGE OVER RAIL & HWY",
    "X540": "STEEL BEAM BRIDGE OVER RAIL & HIGHWAY",
    "X581": "PRESTRESSED BRIDGE OVER RAIL & HWY",
    "X600": "STEEL GIRDER OVER WATER, RAIL & HWY",
    "X620": "STEEL BEAM BRIDGE OVER WATER, RAIL & HWY",
    "X700": "CONCRETE ARCH SECTIONS",
    "X999": "HIGHWAY TUNNEL",
    "Y007": "MINOR STRUCTURE (STORM SEWER, CULVERT)",
    "X033": "STEEL GIRDER BRIDGE OVER WATER & HWY",
}

# Federal Bridge Items FY25

In [6]:
import numpy as np

# ---------------------------------------------------------
# 2. BASE DATA PREPARATION
# ---------------------------------------------------------
# We use 'initial_bridge_list' and 'bridge_types' from your previous verified cells.
df_base = initial_bridge_list.copy()

# Apply the Bridge Type mapping
df_base['Bridge Type Descr'] = df_base['TYPE'].map(bridge_types)

# Calculate Bridge Area
df_base['LENGTH'] = pd.to_numeric(df_base['LENGTH'], errors='coerce').fillna(0)
df_base['WIDTH'] = pd.to_numeric(df_base['WIDTH'], errors='coerce').fillna(0)
df_base['Bridge Area (Sq Ft)'] = df_base['LENGTH'] * df_base['WIDTH']

# Extract the target sections and format as pure integers for the SQL IN clause
raw_sections = df_base['PROPOSALSECTION_ID'].dropna().unique()
target_sections = tuple(int(float(x)) for x in raw_sections)

# Formatting the tuple for the SQL IN clause (handle single item tuple edge case)
if len(target_sections) == 1:
    sections_str = f"({target_sections[0]})"
else:
    sections_str = str(target_sections)

# ---------------------------------------------------------
# 3. EXPLICIT SQL QUERY & DATA FETCH
# ---------------------------------------------------------
# Based on the exact schema structure of the PREC tables you verified, 
# we can explicitly map out the join logic without relying on wildcards.
sql_items = text(f"""
    SELECT 
        pi.PROPOSALSECTION_ID,
        ri.ITEMCLASS,
        pi.PROPOSALITEM_LINENUM,
        ri.REFITEM_NM,
        ri.UNIT,
        ri.DESCR AS [Item Description],
        pi.QTY,
        bid.BIDPRICE,
        bid.EXTENDEDAMOUNT AS Work_Line_Cost,
        v.VENDORNAME
    FROM dbo.PRECON_PROPOSALITEM pi
    INNER JOIN dbo.PRECON_BID bid ON pi.PROPOSALITEM_ID = bid.PROPOSALITEM_ID
    INNER JOIN dbo.PRECON_PROPOSALVENDOR pv ON bid.PROPOSALVENDOR_ID = pv.PROPOSALVENDOR_ID
    LEFT JOIN dbo.PRECON_REFVENDOR v ON pv.REFVENDOR_ID = v.REFVENDOR_ID
    LEFT JOIN dbo.PRECON_REFITEM ri ON pi.REFITEM_ID = ri.REFITEM_ID
    WHERE pv.AWARDED = 1
    AND pi.PROPOSALSECTION_ID IN {sections_str}
""")

try:
    with engine.connect() as conn:
        print("--- FETCHING ITEMIZED BID DATA ---")
        df_items_sql = pd.read_sql(sql_items, conn)
        print(f"SQL Query returned {len(df_items_sql)} rows before merge.")

    # ---------------------------------------------------------
    # 4. DATA CLEANUP & SPEC BOOK FALLBACK
    # ---------------------------------------------------------
    # Tim noted ITEMCLASS can get messy for supplemental specifications. 
    # If the class is missing, we derive it from ODOT Spec Book item prefixes.
    def derive_itemclass(row):
        val = str(row.get('ITEMCLASS', '')).strip()
        
        # If the database successfully provides a class, use it (strip numeric prefixes if present)
        if val and val.upper() not in ['NAN', 'NONE', '']:
            parts = val.split()
            val = parts[-1].upper() if len(parts) > 1 and parts[0].isdigit() else val.upper()
            if val != '':
                return val
                
        # FALLBACK: ODOT Spec Book item prefixes (first 3 digits)
        ref_item = str(row.get('REFITEM_NM', '')).strip().upper()
        if ref_item.startswith('202'): return 'RMVL'   # Removals
        if ref_item.startswith('203'): return 'ERTH'   # Earthwork
        if ref_item.startswith('507'): return 'SPLS'   # Piles
        if ref_item.startswith('509'): return 'SRBR'   # Rebar
        if ref_item.startswith('511'): return 'SCON'   # Concrete
        if ref_item.startswith('513'): return 'SSTL'   # Steel
        if ref_item.startswith('514'): return 'BRPT'   # Bridge Paint
        if ref_item.startswith('526'): return 'SCON'   # Approach Slabs
        if ref_item.startswith('601'): return 'EC'     # Erosion Control
        if ref_item.startswith('607'): return 'FENC'   # Fencing
        if ref_item.startswith('611') or ref_item.startswith('613'): return 'DRNG' # Drainage
        if ref_item.startswith('625'): return 'LTNG'   # Lighting
        if ref_item.startswith('840') or ref_item.startswith('867'): return 'MISC'
        if ref_item.startswith('863'): return 'ERTH'
        
        # Keyword Fallback just in case
        desc = str(row.get('Item Description', '')).strip().upper()
        if 'REMOV' in desc: return 'RMVL'
        if 'ROCK CHANNEL' in desc or 'SLOPE PROTECTION' in desc or 'FILTER' in desc: return 'EC'
        
        return 'SOTH' # General Structure Other default
        
    df_items_sql['ITEMCLASS'] = df_items_sql.apply(derive_itemclass, axis=1)

    # ---------------------------------------------------------
    # 5. MERGE AND APPLY FEDERAL COST LOGIC
    # ---------------------------------------------------------
    # Bulletproof the merge keys by forcing both to clean string integers
    df_base['PROPOSALSECTION_ID_CLEAN'] = pd.to_numeric(df_base['PROPOSALSECTION_ID'], errors='coerce').fillna(0).astype(int).astype(str)
    df_items_sql['PROPOSALSECTION_ID_CLEAN'] = pd.to_numeric(df_items_sql['PROPOSALSECTION_ID'], errors='coerce').fillna(0).astype(int).astype(str)
    
    federal_bridge_items = pd.merge(df_base, df_items_sql, on='PROPOSALSECTION_ID_CLEAN', how='inner')
    print(f"Rows after merge: {len(federal_bridge_items)}")

    # Logic: Removals (RMVL), Erosion Control (EC), and Drainage (DRNG) are usually 
    # not participating in this Federal Cost calculation.
    non_participating_classes = ['RMVL', 'EC', 'DRNG']

    def calc_fed_cost(row):
        item_class = str(row['ITEMCLASS']).strip().upper()
        if item_class in non_participating_classes:
            return np.nan  # Leave blank (NaN) just like the Excel sheet
        return row['Work_Line_Cost']

    federal_bridge_items['Fed_Work_Line_Cost'] = federal_bridge_items.apply(calc_fed_cost, axis=1)
    federal_bridge_items['Comments'] = ''

    # ---------------------------------------------------------
    # 6. FINAL FORMATTING
    # ---------------------------------------------------------
    final_columns = [
        'LETTING_NM', 'PROPOSAL_NM', 'CONTRACTALTERNATE_NM1', 'PID', 'FEDPROJECTNUM',
        'AWARDEDAMOUNT', 'VENDORNAME', 'Section Desc', 'Existing SFN', 'New SFN',
        'FEDWORKCLASS', 'BRIDGE_NM', 'DESCR', 'Bridge Type Descr', 'Bridge Area (Sq Ft)',
        'PROPOSALSECTION_NM', 'ITEMCLASS', 'PROPOSALITEM_LINENUM', 'REFITEM_NM', 'UNIT',
        'Item Description', 'QTY', 'BIDPRICE', 'Work_Line_Cost', 'Fed_Work_Line_Cost', 'Comments'
    ]

    for col in final_columns:
        if col not in federal_bridge_items.columns:
            federal_bridge_items[col] = ''

    federal_bridge_items = federal_bridge_items[final_columns].rename(columns={
        'AWARDEDAMOUNT': 'Award Amt',
        'DESCR': 'Bridge Descr'
    })

    # Sort to match the typical view (Letting Date/Proposal/Line Num)
    federal_bridge_items = federal_bridge_items.sort_values(['LETTING_NM', 'PROPOSAL_NM', 'PROPOSALITEM_LINENUM'])

    print(f"--- FEDERAL BRIDGE ITEMS GENERATED ---")
    print(f"Total Line Items Extracted: {len(federal_bridge_items)}")
    print("\nPreview of the Items Sheet (Note the blanks/NaNs in Fed_Work_Line_Cost!):")
    
    pd.options.display.float_format = '{:,.2f}'.format
    print(federal_bridge_items[['PROPOSAL_NM', 'BRIDGE_NM', 'ITEMCLASS', 'Work_Line_Cost', 'Fed_Work_Line_Cost']].head(15))

except Exception as e:
    print(f"Error processing item data: {e}")

--- FETCHING ITEMIZED BID DATA ---
SQL Query returned 2200 rows before merge.
Rows after merge: 2396
--- FEDERAL BRIDGE ITEMS GENERATED ---
Total Line Items Extracted: 2396

Preview of the Items Sheet (Note the blanks/NaNs in Fed_Work_Line_Cost!):
   PROPOSAL_NM     BRIDGE_NM ITEMCLASS  Work_Line_Cost  Fed_Work_Line_Cost
22   FUL101140  FUL-120-1408      RMVL      200,000.00                 NaN
9    FUL101140  FUL-120-1408      RMVL        7,600.00                 NaN
3    FUL101140  FUL-120-1408      SOTH    1,000,000.00        1,000,000.00
4    FUL101140  FUL-120-1408      SOTH      178,640.00          178,640.00
5    FUL101140  FUL-120-1408      SOTH       35,000.00           35,000.00
7    FUL101140  FUL-120-1408      SPLS      298,410.00          298,410.00
8    FUL101140  FUL-120-1408      SPLS       18,860.00           18,860.00
11   FUL101140  FUL-120-1408      SPLS      250,425.00          250,425.00
10   FUL101140  FUL-120-1408      SRBR      161,718.00          161,718.00
1 

# //TODO - Seems to be working, but returning 2x expected values, 2396 here, 1025 in excel

# Federal Bridge List FY25

In [7]:
# ---------------------------------------------------------
# 2. BASE DATA & CALCULATIONS
# ---------------------------------------------------------
# Use the initial_bridge_list generated from the first cell
df_list = initial_bridge_list.copy()

# Clean up SFNs early (replace string 'None' or 'nan' with actual blanks for cleaner Excel export)
df_list['Existing SFN'] = df_list['Existing SFN'].astype(str).str.strip()
df_list['Existing SFN'] = df_list['Existing SFN'].replace({'nan': '', 'None': '', 'none': '', '<NA>': ''})

# --- CRITICAL FIX: THE WHITELIST FILTER ---
# Tim's bridge_types dictionary isn't just for naming; it acts as a whitelist!
# By mapping the types and dropping NaNs, we automatically filter out retaining walls, 
# noise walls, and generic approach work that snuck into the Category 14/15 pull, 
# WHILE keeping the 3 valid bridges that happen to have 0 Area or missing SFNs.
df_list['Bridge Type Descr'] = df_list['TYPE'].map(bridge_types)
df_list = df_list[df_list['Bridge Type Descr'].notna()]

# Calculate Bridge Area BEFORE deduplicating so we can keep the row with actual dimensions
df_list['LENGTH'] = pd.to_numeric(df_list['LENGTH'], errors='coerce').fillna(0)
df_list['WIDTH'] = pd.to_numeric(df_list['WIDTH'], errors='coerce').fillna(0)
df_list['Bridge Area (Sq Ft)'] = df_list['LENGTH'] * df_list['WIDTH']

# Sort by Area (Descending) to ensure we keep the section with actual bridge dimensions
df_list = df_list.sort_values(by=['PROPOSAL_NM', 'Existing SFN', 'Bridge Area (Sq Ft)'], ascending=[True, True, False])

# Deduplicate strictly by Proposal and Existing SFN to get exactly one row per physical bridge.
df_list = df_list.drop_duplicates(subset=['PROPOSAL_NM', 'Existing SFN'])

# ---------------------------------------------------------
# 3. FETCH SMS BRIDGE DATA (NHS & DEFICIENCY)
# ---------------------------------------------------------
# Extract unique 'Existing SFN' values to query the SMS_BRIDGE table
sfns = [sfn for sfn in df_list['Existing SFN'].unique() if sfn != '']

# Format for SQL IN clause
if len(sfns) == 1:
    sfn_str = f"('{sfns[0]}')"
elif len(sfns) > 1:
    sfn_str = str(tuple(sfns))
else:
    sfn_str = "('')"

sql_sms = text(f"""
    SELECT 
        SFN,
        INVENT_NHS_CD,
        DEFIC_FUNC_RATING
    FROM dbo.SMS_BRIDGE
    WHERE SFN IN {sfn_str}
""")

try:
    with engine.connect() as conn:
        print("--- FETCHING SMS BRIDGE DATA ---")
        df_sms = pd.read_sql(sql_sms, conn)
        
    # Clean SFN in the SMS dataframe and drop any potential duplicates 
    # (in case SMS_BRIDGE retains historical records)
    df_sms['SFN'] = df_sms['SFN'].astype(str).str.strip()
    df_sms = df_sms.drop_duplicates(subset=['SFN'])

    # ---------------------------------------------------------
    # 4. MERGE AND FORMAT FINAL LIST
    # ---------------------------------------------------------
    # Merge the SMS data into the bridge list
    federal_bridge_list = pd.merge(
        df_list, 
        df_sms, 
        left_on='Existing SFN', 
        right_on='SFN', 
        how='left'
    )
    
    # Drop the duplicate SFN column from the merge
    if 'SFN' in federal_bridge_list.columns:
        federal_bridge_list = federal_bridge_list.drop(columns=['SFN'])

    # Define final column order based on the Excel screenshot
    final_cols = [
        'LETTING_DT', 'LETTING_NM', 'PROPOSAL_NM', 'PID', 'CONTRACTALTERNATE_NM1',
        'AWARDEDAMOUNT', 'FEDPROJECTNUM', 'PROPOSALSECTION_NM', 'Section Desc',
        'Existing SFN', 'New SFN', 'FEDWORKCLASS', 'BRIDGE_NM', 'DESCR', 'TYPE',
        'LENGTH', 'WIDTH', 'NUMOFSPANS', 'Modified', 'Flag for Removal',
        'Bridge Type Descr', 'Bridge Area (Sq Ft)', 'INVENT_NHS_CD', 'DEFIC_FUNC_RATING'
    ]

    # Ensure all columns exist, fill missing with blank
    for col in final_cols:
        if col not in federal_bridge_list.columns:
            federal_bridge_list[col] = ''

    federal_bridge_list = federal_bridge_list[final_cols]
    
    # Sort for review
    federal_bridge_list = federal_bridge_list.sort_values(['LETTING_DT', 'PROPOSAL_NM'])

    print(f"--- FEDERAL BRIDGE LIST GENERATED ---")
    print(f"Total Bridges: {len(federal_bridge_list)}")
    print("\nPreview of the Federal Bridge List:")
    pd.options.display.float_format = '{:,.2f}'.format
    print(federal_bridge_list[['PROPOSAL_NM', 'Existing SFN', 'Bridge Area (Sq Ft)', 'INVENT_NHS_CD', 'DEFIC_FUNC_RATING']].head(15))

except Exception as e:
    print(f"Error processing SMS Bridge data: {e}")

--- FETCHING SMS BRIDGE DATA ---
--- FEDERAL BRIDGE LIST GENERATED ---
Total Bridges: 36

Preview of the Federal Bridge List:
   PROPOSAL_NM Existing SFN  Bridge Area (Sq Ft) INVENT_NHS_CD  \
15   FUL101140      2601745             2,194.20             0   
4    CLA102707      1202138             2,139.72             0   
8     CUY13184      1801929             5,385.15             0   
17   GEA111000      2800756             3,237.38             0   
18   GEA115823      2801507             3,629.32             0   
24   LOG114937      4602315             1,950.12             0   
35   WAR112975      8305161             6,452.02             0   
31   PIK106540      6630839             2,464.00             0   
10   DEL105433      2103575             2,320.00             0   
11   DEL105433      2103664             2,312.00             0   
19   HAM102790      3106632             4,962.48             1   
22   LIC104981      4501810             4,140.80             0   
23   LIC104981  

In [8]:
# ==============================================================================
# OHIO DOT BRIDGE REPLACEMENT - BRIDGE COST TOTALS
# Objective: Recreate the "Bridge Costs Totals" sheet by merging the clean 
# Federal Bridge List with the aggregated costs from the Federal Bridge Items.
# ==============================================================================

import pandas as pd

# ---------------------------------------------------------
# 1. AGGREGATE ITEMIZED COSTS
# ---------------------------------------------------------
# Use the federal_bridge_items dataframe generated in Step 2.
# We group by Proposal and Bridge Name to sum the costs.
# Note: pandas .sum() automatically treats the NaNs in Fed_Work_Line_Cost as 0!

df_items = federal_bridge_items.copy()

# Ensure numeric types before summing
df_items['Work_Line_Cost'] = pd.to_numeric(df_items['Work_Line_Cost'], errors='coerce').fillna(0)
df_items['Fed_Work_Line_Cost'] = pd.to_numeric(df_items['Fed_Work_Line_Cost'], errors='coerce').fillna(0)

# Aggregate the costs
cost_totals = df_items.groupby(['PROPOSAL_NM', 'BRIDGE_NM']).agg({
    'Work_Line_Cost': 'sum',
    'Fed_Work_Line_Cost': 'sum'
}).reset_index()

# Rename columns to match the Pivot Table output
cost_totals = cost_totals.rename(columns={
    'Work_Line_Cost': 'Sum of Work_Line_Cost',
    'Fed_Work_Line_Cost': 'Sum of Fed_Work_Line_Cost'
})

# ---------------------------------------------------------
# 2. MERGE COSTS ONTO THE BRIDGE LIST
# ---------------------------------------------------------
# Use the deduplicated federal_bridge_list (36 bridges) generated in Step 3.
df_list = federal_bridge_list.copy()

# Merge the aggregated costs onto the bridge list
final_totals = pd.merge(
    df_list,
    cost_totals,
    on=['PROPOSAL_NM', 'BRIDGE_NM'],
    how='left'
)

# Fill any missing costs with 0 just in case a bridge had no billable items
final_totals['Sum of Work_Line_Cost'] = final_totals['Sum of Work_Line_Cost'].fillna(0)
final_totals['Sum of Fed_Work_Line_Cost'] = final_totals['Sum of Fed_Work_Line_Cost'].fillna(0)

# ---------------------------------------------------------
# 3. CALCULATE FINAL METRICS
# ---------------------------------------------------------
# Calculate Unit Cost (Cost per Sq Ft) using the FEDERAL work line cost.
# We use a lambda function to safely handle division by zero for any 
# rare bridges that might have 0 Area.
final_totals['Cost per Sq Ft'] = final_totals.apply(
    lambda row: row['Sum of Fed_Work_Line_Cost'] / row['Bridge Area (Sq Ft)'] if row['Bridge Area (Sq Ft)'] > 0 else 0,
    axis=1
)

# Select and order the final columns exactly as Tim reviews them
final_columns = [
    'LETTING_DT', 'LETTING_NM', 'PROPOSAL_NM', 'PID', 'CONTRACTALTERNATE_NM1',
    'AWARDEDAMOUNT', 'FEDPROJECTNUM', 'PROPOSALSECTION_NM', 'Section Desc',
    'Existing SFN', 'New SFN', 'FEDWORKCLASS', 'BRIDGE_NM', 'DESCR', 'TYPE',
    'LENGTH', 'WIDTH', 'NUMOFSPANS', 'Modified', 'Flag for Removal',
    'Bridge Type Descr', 'Bridge Area (Sq Ft)', 'INVENT_NHS_CD', 'DEFIC_FUNC_RATING',
    'Sum of Work_Line_Cost', 'Sum of Fed_Work_Line_Cost', 'Cost per Sq Ft'
]

# Ensure all columns exist to prevent KeyError
for col in final_columns:
    if col not in final_totals.columns:
        final_totals[col] = ''

final_totals = final_totals[final_columns]

# Sort alphabetically by Proposal and Bridge Name for Tim's review
final_totals = final_totals.sort_values(['PROPOSAL_NM', 'BRIDGE_NM'])

# ---------------------------------------------------------
# 4. OUTPUT PREVIEW & EXPORT
# ---------------------------------------------------------
pd.options.display.float_format = '{:,.2f}'.format

print(f"--- FINAL BRIDGE COST TOTALS REPLICATED ---")
print(f"Total Unique Bridges: {len(final_totals)}")

print("\nPreview of the Final Pivot Table:")
print("Notice the difference between Work Cost and Fed Work Cost where items were excluded!\n")
print(final_totals[['PROPOSAL_NM', 'BRIDGE_NM', 'Bridge Area (Sq Ft)', 'Sum of Work_Line_Cost', 'Sum of Fed_Work_Line_Cost', 'Cost per Sq Ft']].head(15))

# Uncomment this line if you want to export the final result straight to Excel!
# final_totals.to_excel('Bridge_Cost_Totals_FY25.xlsx', index=False)

--- FINAL BRIDGE COST TOTALS REPLICATED ---
Total Unique Bridges: 36

Preview of the Final Pivot Table:
Notice the difference between Work Cost and Fed Work Cost where items were excluded!

   PROPOSAL_NM         BRIDGE_NM  Bridge Area (Sq Ft)  Sum of Work_Line_Cost  \
33   ALL117742       SFN 0203484             1,444.00             572,398.20   
32   ATH119810   ATH-MADSN-00400             2,996.79           1,552,937.50   
30   BEL117373  BEL-C0004-27.460             2,490.00           1,012,554.80   
35   BRO100897     BRO-52-12.452            12,075.67           4,162,897.77   
1    CLA102707       CLA-42-0820             2,139.72             637,184.04   
34   CLI105224      CLI-134-0225             4,586.80           1,199,975.20   
21   CUY104132       CUY-14-0693            34,396.54          21,591,451.80   
22   CUY104132    CUY-CR240-0061             3,360.00           2,744,377.01   
2     CUY13184           1801930             5,385.15           7,593,471.00   
18   DEF11